# 🖼️ AI 이미지 생성 — 포스터, 앨범 커버, 무대 컨셉을 만들어보세요

텍스트로 이미지를 만들거나, 기존 이미지를 새로운 스타일로 변형할 수 있습니다.
공연 포스터, 앨범 커버, 무대 컨셉 이미지 등에 활용해보세요.

아래 세 가지 도구 중 **원하는 것만 골라서** 실행하세요.

| 도구 | 입력 | 특징 |
|------|------|------|
| **Flux schnell** | 텍스트 | 최고 품질, 4스텝 생성 (4bit 양자화로 T4 적합) |
| **SD 3.5 Medium** | 텍스트 | Stability AI 최신 모델, 세밀 조정 가능 |
| **SDXL-Turbo img2img** | 이미지 + 텍스트 | 기존 이미지를 2스텝만에 스타일 변형 |

▶ 먼저 공통 패키지를 설치합니다.

In [ ]:
%%capture
!pip install -q diffusers transformers accelerate bitsandbytes

import warnings
warnings.filterwarnings('ignore')

print('✅ 공통 패키지 설치 완료!')

---
# 옵션 A: Flux schnell — 최고 품질 이미지 생성

Black Forest Labs의 **FLUX.1-schnell**은 현재 오픈소스 이미지 생성 모델 중 최고 품질입니다.
단 **4스텝**만에 이미지를 생성합니다. (Apache 2.0 라이선스, 무료)

원래 ~23GB VRAM이 필요하지만, **4bit 양자화(quantization)**로 T4에서도 실행 가능합니다.

⏰ 모델 로딩 2~3분 + 이미지 생성 약 30초

### A-1. 모델 로드 (4bit 양자화)

▶ 아래 셀을 실행하세요. 모델 다운로드 + 양자화에 2~3분 걸립니다.

In [ ]:
import torch

print('🔄 Flux schnell 모델을 4bit 양자화로 불러오는 중...')
print('   (원래 23GB → 양자화로 ~8GB, T4에서 실행 가능)')

try:
    from diffusers import FluxPipeline
    from transformers import T5EncoderModel, BitsAndBytesConfig

    model_id = 'black-forest-labs/FLUX.1-schnell'

    # T5 텍스트 인코더 4bit 양자화
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    )
    text_encoder_2 = T5EncoderModel.from_pretrained(
        model_id,
        subfolder='text_encoder_2',
        quantization_config=quant_config,
        torch_dtype=torch.float16,
    )

    pipe_flux = FluxPipeline.from_pretrained(
        model_id,
        text_encoder_2=text_encoder_2,
        torch_dtype=torch.float16,
    )
    pipe_flux.enable_model_cpu_offload()

    print('✅ Flux schnell (4bit) 로딩 완료!')
    flux_available = True
except Exception as e:
    print(f'⚠️ Flux 로드 실패: {e}')
    print('💡 옵션 B (SD 3.5 Medium)를 사용해보세요.')
    flux_available = False

### A-2. 프롬프트 입력 → 이미지 생성

▶ 만들고 싶은 이미지를 영어로 설명하세요.

**프롬프트 예시:**

| 프롬프트 | 느낌 |
|---------|------|
| `minimalist concert poster, solo pianist on dark stage, single spotlight` | 미니멀 공연 포스터 |
| `watercolor painting of piano keys transforming into ocean waves` | 수채화 앨범 커버 |
| `abstract digital art inspired by Chopin nocturne, deep blue and gold` | 추상 디지털 아트 |

In [ ]:
import matplotlib.pyplot as plt
import os

if flux_available:
    prompt = "minimalist concert poster, solo pianist on dark stage, single spotlight"  # ← 여기를 수정하세요

    print(f'🎨 프롬프트: "{prompt}"')
    print('🔄 이미지를 생성하고 있습니다...')

    image = pipe_flux(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        width=512, height=512,
    ).images[0]

    output_file = 'flux_output.png'
    image.save(output_file)

    print('✅ 생성 완료!')
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(prompt[:60] + '...', fontsize=10)
    plt.show()

    from google.colab import files
    files.download(output_file)
else:
    print('⚠️ Flux가 로드되지 않았습니다. 옵션 B를 사용하세요.')

---
# 옵션 B: SD 3.5 Medium — Stability AI 최신 모델

**Stable Diffusion 3.5 Medium**은 Stability AI의 최신 이미지 생성 모델입니다.
SD 1.5보다 훨씬 높은 품질과 텍스트 이해력을 가지고 있습니다.

**핵심 파라미터:**
- `guidance_scale`: 프롬프트에 얼마나 충실할지 (4~8 권장)
- `num_inference_steps`: 생성 단계 수 (28 권장)
- `negative_prompt`: 원하지 않는 요소

⏰ 이미지 1장 생성에 약 1분

### B-1. 모델 로드 → 프롬프트 입력 → 생성

▶ 프롬프트를 수정하고 셀을 실행하세요. 모델 로딩에 1~2분 걸립니다.

In [ ]:
import torch
import gc
import matplotlib.pyplot as plt

# 이전 모델 메모리 해제
if 'pipe_flux' in dir():
    del pipe_flux
if 'text_encoder_2' in dir():
    del text_encoder_2
gc.collect()
torch.cuda.empty_cache()

print('🔄 SD 3.5 Medium 모델을 불러오는 중...')

from diffusers import StableDiffusion3Pipeline

pipe_sd35 = StableDiffusion3Pipeline.from_pretrained(
    'stabilityai/stable-diffusion-3.5-medium',
    torch_dtype=torch.float16,
)
pipe_sd35.enable_model_cpu_offload()  # T4 메모리 최적화
print('✅ SD 3.5 Medium 로딩 완료!')

# === 여기를 수정하세요 ===
prompt = "watercolor painting of piano keys transforming into ocean waves"  # ← 프롬프트
negative_prompt = "blurry, low quality, text, watermark"  # ← 원하지 않는 요소
guidance_scale = 7.0   # ← 조정해보세요 (4~8)
num_inference_steps = 28  # ← 조정해보세요 (20~40)
# ========================

print(f'🎨 프롬프트: "{prompt}"')
print('🔄 이미지를 생성하고 있습니다...')

image = pipe_sd35(
    prompt=prompt,
    negative_prompt=negative_prompt,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    width=512, height=512,
).images[0]

output_file = 'sd35_output.png'
image.save(output_file)

print('✅ 생성 완료!')
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.show()

from google.colab import files
files.download(output_file)

---
# 옵션 C: SDXL-Turbo img2img — 기존 이미지를 새로운 스타일로

본인의 사진이나 그림을 넣고, AI가 **단 2스텝**만에 새로운 스타일로 변형합니다.
SDXL-Turbo는 고속 모델이라 결과를 바로 확인할 수 있습니다.

**핵심 파라미터 — `strength`:**
- `0.3` → 원본이 거의 유지됨 (살짝 변형)
- `0.7` → 원본 구도는 남지만 스타일 변형
- `1.0` → 원본 거의 무시, 완전 새로 생성

⏰ 이미지 변형에 약 5초

### C-1. 이미지 업로드

▶ 변형할 이미지를 업로드하세요.

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
input_image_path = list(uploaded.keys())[0]

input_image = Image.open(input_image_path).convert('RGB').resize((512, 512))

print(f'✅ 이미지 업로드: {input_image_path}')
plt.figure(figsize=(6, 6))
plt.imshow(input_image)
plt.axis('off')
plt.title('원본 이미지')
plt.show()

### C-2. 스타일 변형

▶ 프롬프트와 strength를 수정하고 실행하세요.

In [ ]:
import torch
import gc
from diffusers import AutoPipelineForImage2Image
import matplotlib.pyplot as plt

# 이전 모델 메모리 해제
for name in ['pipe_sd35', 'pipe_flux', 'text_encoder_2']:
    if name in dir():
        exec(f'del {name}')
gc.collect()
torch.cuda.empty_cache()

print('🔄 SDXL-Turbo img2img 모델을 불러오는 중...')
pipe_img2img = AutoPipelineForImage2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe_img2img = pipe_img2img.to('cuda')
print('✅ SDXL-Turbo 로딩 완료!')

# === 여기를 수정하세요 ===
prompt = "watercolor painting, artistic, beautiful colors"  # ← 원하는 스타일
strength = 0.7  # ← 0~1 사이. 높을수록 크게 변형
# ========================

print(f'🎨 프롬프트: "{prompt}" (strength: {strength})')
print('🔄 스타일 변형 중...')

result = pipe_img2img(
    prompt=prompt,
    image=input_image,
    strength=strength,
    guidance_scale=0.0,  # SDXL-Turbo: guidance 불필요
    num_inference_steps=2,  # 2스텝만에 완료
).images[0]

output_file = 'img2img_output.png'
result.save(output_file)

print('✅ 변형 완료!')

# 원본과 결과 나란히
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(input_image)
axes[0].set_title('원본')
axes[0].axis('off')
axes[1].imshow(result)
axes[1].set_title(f'변형 (strength={strength})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

from google.colab import files
files.download(output_file)